In [ ]:
import pandas as pd
import numpy as np


df = pd.read_csv(f"credit_risk_dataset.csv")

# df[...] : devuelve solo las filas donde el valor es TRUE. Es un filtro


df_limpio = df.dropna()


print("Filas antes:", len(df))
print("Filas después:", len(df_limpio))
print("Filas eliminadas:", len(df) - len(df_limpio))

print(df_limpio[['person_age', 'person_emp_length', 'person_income']].describe())



# Filtro 1: eliminar las edades imposibles (> 100 años)

df_limpio = df_limpio[df_limpio['person_age'] <= 100]

# Filtro 2: la antiguedad laboral no puede superar (edad - 18)
df_limpio = df_limpio[df_limpio['person_emp_length'] <= df_limpio['person_age'] - 18]

print("Filas finales:" , len(df_limpio))
print(df_limpio[['person_age', 'person_emp_length', 'person_income']].describe())

# Descripcion del grafo propuesto: 

df_edges = df_limpio[
    ('person_age', 'loan_intent'),                   # A → E
    ('person_age', 'person_income'),                 # A → B
    ('person_age', 'cb_person_cred_hist_length'),    # A → L
    ('person_age', 'person_emp_length'),             # A → D
    ('loan_intent', 'person_home_ownership'),        # E → C
    ('person_income', 'loan_percent_income'),        # B → J
    ('person_income', 'person_home_ownership'),      # B → C
    ('person_income', 'loan_amnt'),                  # B → G
    ('person_income', 'loan_int_rate'),              # B → H
    ('person_income', 'loan_grade'),                 # B → F
    ('person_income', 'cb_person_default_on_file'),  # B → K
    ('person_emp_length', 'person_home_ownership'),  # D → C
    ('loan_grade', 'loan_status'),                   # F → I
    ('loan_int_rate', 'loan_status'),                # H → I
    ('loan_percent_income', 'loan_status'),          # J → I
    ('cb_person_default_on_file', 'loan_status'),    # K → I
    ('loan_amnt', 'loan_status'),                    # G → I
]


Filas antes: 32581
Filas después: 28638
Filas eliminadas: 3943
         person_age  person_emp_length  person_income
count  28638.000000       28638.000000   2.863800e+04
mean      27.727216           4.788672   6.664937e+04
std        6.310441           4.154627   6.235645e+04
min       20.000000           0.000000   4.000000e+03
25%       23.000000           2.000000   3.948000e+04
50%       26.000000           4.000000   5.595600e+04
75%       30.000000           7.000000   8.000000e+04
max      144.000000         123.000000   6.000000e+06
Filas finales: 21568
         person_age  person_emp_length  person_income
count  21568.000000       21568.000000   2.156800e+04
mean      28.660979           3.475473   6.514939e+04
std        6.577067           3.310146   5.417122e+04
min       20.000000           0.000000   4.000000e+03
25%       24.000000           1.000000   3.720000e+04
50%       27.000000           3.000000   5.400000e+04
75%       32.000000           5.000000   7.800000e+0

### Parte 1 - Red Bayesiana

Las variables a ocupar son:

A = person_age: Edad del solicitante
B = person_income: Ingreso anual 
C = person_home_ownership: Tipo de tenencia de vivienda (arrienda / hipoteca / propia / otro)  
D = person_emp_length: Años de antigüedad laboral  
E = loan_intent: Propósito del préstamo (educación, salud, personal, etc.)  
F = loan_grade: Grado del préstamo según solvencia. A = alta solvencia / bajo riesgo; G = la más baja / mayor riesgo  
G = loan_amnt: Monto solicitado  
H = loan_int_rate: Tasa de interés del préstamo  
I = loan_status: Estado: 0 = no default, 1 = default  
J = loan_percent_income: Monto del préstamo como % del ingreso  
K = cb_person_default_on_file: Si tiene un default histórico registrado (Y/N)  
L = cb_person_cred_hist_length: Largo del historial crediticio (años)  


JUSTIFICAR DEPENDENCIAS

edges = [
    ('person_age', 'loan_intent'),                   # A → E
    ('person_age', 'person_income'),                 # A → B
    ('person_age', 'cb_person_cred_hist_length'),    # A → L
    ('person_age', 'person_emp_length'),             # A → D
    ('loan_intent', 'person_home_ownership'),        # E → C
    ('person_income', 'loan_percent_income'),        # B → J
    ('person_income', 'person_home_ownership'),      # B → C
    ('person_income', 'loan_amnt'),                  # B → G
    ('person_income', 'loan_int_rate'),              # B → H
    ('person_income', 'loan_grade'),                 # B → F
    ('person_income', 'cb_person_default_on_file'),  # B → K
    ('person_emp_length', 'person_home_ownership'),  # D → C
    ('loan_grade', 'loan_status'),                   # F → I
    ('loan_int_rate', 'loan_status'),                # H → I
    ('loan_percent_income', 'loan_status'),          # J → I
    ('cb_person_default_on_file', 'loan_status'),    # K → I
    ('loan_amnt', 'loan_status'),                    # G → I
]

A → {E, B, L, D}
B → {J, C, G, H, F, K}
E → C, D → C
{F, H, J, K, G} → I




Necesitamos que cada variable tenga un número finito y pequeño de categorías. 
El problema actual con el dataset es que algunas columnas tienen numeros continuos que podrian tomar infinitos valores
Por ende, se discretizaran: tomar las variables continuas y ingresarlas en una "caja" (bins); que para este caso, cada bin tendra aproximadamente la misma cantidad de gente  
Esto sera hecho diviendo los datos en cuantiles:
  - si se divide la variable "person_income" en 4 cajas, cada caja tiene un 25% de los datos. 

Estas seran las variables a discretizar(que son numéricas): 
person_age
person_income
person_emp_length
loan_amnt
loan_int_rate
loan_percent_income
cb_person_cred_hist_length


